# ⚡ PLAN 2: FAST — Dual-Resident on Tesla T4 (16GB VRAM • 65–75 tok/s • ~1.53 CU/h)

> **When to Use:** Designed for end-of-month sessions when Compute Units are running low ($< 35\text{ CU}$) or when maximum responsiveness (65–75 tok/s) is required at minimal cost (**~1.53 CU/h** on Tesla T4).
> **Dual-Resident Mechanism (`OLLAMA_MAX_LOADED_MODELS="2"`):** Concurrently pins both **`deepseek-r1:8b`** (5.2 GB) and **`qwen2.5-coder:7b` ⭐** (4.5 GB) into T4 16GB VRAM. **Switch latency = 0.0s** (zero model swapping).

---

## 🧩 Dual-Resident Model Configuration (Tesla T4 16GB)

| Role | Model | VRAM & Specs |
| :--- | :--- | :--- |
| 🎼 **Conductor (Planning)** | **`deepseek-r1:8b`** | ~5.2 GB VRAM • Fast CoT `<think>` reasoning (~55 tok/s), resident in VRAM. |
| 🛠️ **Worker (Code / Work / Diagram)** | **`qwen2.5-coder:7b` ⭐** *(or `qwen3-coder:8b`)* | ~4.5 GB VRAM • 65–75 tok/s generation speed (or `qwen3-coder:8b` ~5.5 GB), resident in VRAM. |

---

## 🛡️ VRAM Parameters & Optimization (80.2% VRAM — 0.0s Switch)
- **Configuration:** `OLLAMA_MAX_LOADED_MODELS="2"` + `OLLAMA_FLASH_ATTENTION="1"` + `OLLAMA_KV_CACHE_TYPE="q8_0"`.
- **Total VRAM (16K context):** `5.2 GB (M1) + 4.5 GB (M2) + 3.0 GB (KV Cache) = 12.7 GB / 15.84 GB (80.2%)`.
- **Characteristics:** 0 model swaps, instant transitions between planning and coding (0.0s latency).

---

💡 **See Also:** [`Plan 1 T4 (14B)`](colab_ollama_plan1_performance_t4.ipynb) • [`Plan 1 A100 (32B)`](colab_ollama_plan1_performance_a100.ipynb) • [`Tutorial (1-Model)`](colab_ollama_tutorial.ipynb)

### Step 1: Hardware & VRAM Verification (NVIDIA GPU)
Verify that the GPU (Tesla T4 16GB / A100 40GB) is ready in the Colab runtime.

In [ ]:
# Step 1: Verify NVIDIA GPU hardware and active VRAM capacity
!nvidia-smi

### Step 2: Model Storage Configuration (Colab NVMe SSD / Google Drive)
> 💡 **Recommendation:** Keep `USE_GDRIVE = False` (default) to use **internal Colab NVMe SSD** (>600 MB/s, zero authorization prompts). Toggle `USE_GDRIVE = True` if running via a web browser and you wish to persist the cache across sessions.

In [ ]:
import os

# Set to True if running in a web browser (Safari/Chrome) and you wish to persist models to Google Drive
# Default is False to prioritize high-speed internal NVMe SSD (>600 MB/s) & 100% compatibility with IDE Kernels
USE_GDRIVE = False

if USE_GDRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        models_dir = '/content/drive/MyDrive/ollama_models'
        os.environ['OLLAMA_MODELS'] = models_dir
        os.makedirs(models_dir, exist_ok=True)
        print(f"✅ Google Drive cache activated at: {models_dir}")
    except Exception as e:
        print(f"ℹ️ Running in IDE Kernel (Skipping Google Drive mount: {e}).")
        print("⚡ Using internal Colab NVMe SSD storage (/root/.ollama/models).")
else:
    print("⚡ Using internal Colab NVMe SSD storage (/root/.ollama/models) — Optimal model loading speed.")

### Step 3: Launch Dual-Resident Ollama Daemon (`OLLAMA_MAX_LOADED_MODELS="2"`)
- **Swapfile:** Create 16GB Linux NVMe Swapfile (expands virtual memory to 28.67 GB).
- **Dual-Residency:** Configure `OLLAMA_MAX_LOADED_MODELS="2"` to pin both models simultaneously in 16GB VRAM.
- **VRAM Optimizations:** Enable Flash Attention (`1`) and quantized KV Cache (`q8_0`) to keep memory footprint at ~12.7 GB / 15.84 GB (80.2%).

In [ ]:
import os
import subprocess
import time
import urllib.request
from IPython.display import clear_output

# 1. Provision 16GB Linux NVMe Swapfile (Expands System RAM from 12.67GB to 28.67GB, 100% OOM protection)
!test -f /swapfile || (fallocate -l 16G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile)

# 2. Install zstd and official Ollama Linux release
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Configure Dual-Resident environment variables on Tesla T4 16GB (80.2% VRAM)
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_KEEP_ALIVE"] = "24h"          # Keep both models resident in VRAM throughout the session
env["OLLAMA_MAX_LOADED_MODELS"] = "2"      # DUAL-RESIDENT BREAKTHROUGH: Co-pin M1 (5.2GB) + M2 (4.5GB) in 16GB VRAM!
env["OLLAMA_FLASH_ATTENTION"] = "1"       # Enable Flash Attention for high token throughput (65-75 tok/s)
env["OLLAMA_KV_CACHE_TYPE"] = "q8_0"      # Quantize KV Cache to 8-bit, keeping 16K ctx KV footprint to ~3.0 GB

# 4. Launch 'ollama serve' in background
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
clear_output(wait=True)
time.sleep(3)

try:
    with urllib.request.urlopen("http://localhost:11434/") as resp:
        print("🎉 Plan 2 Fast Dual-Resident Daemon ready on T4 (MAX_LOADED_MODELS=2 | FLASH_ATTENTION=1 | KV=q8_0):")
        print(resp.read().decode("utf-8"))
except Exception as e:
    print("⚠️ Failed to connect to Ollama daemon:", e)

### Step 4: Pull & Co-Pin Both Models in VRAM
- **Model 1 (Planning):** `deepseek-r1:8b` (~5.2 GB VRAM, `<think>` reasoning ~55 tok/s).
- **Model 2 (Coding):** `qwen2.5-coder:7b` ⭐ (~4.5 GB VRAM, 65–75 tok/s). Toggle `USE_QWEN3_CODER = True` to use `qwen3-coder:8b` (~5.5 GB).
- Both models are pre-loaded into VRAM for instantaneous transitions (**0.0s switch latency**).

In [ ]:
import json
import urllib.request
from IPython.display import clear_output

# ==============================================================================
# 🎛️ 1-LINE TOGGLE FOR MODEL 2 (ULTRA-FAST CODING SPECIALIST ON T4):
# - False (Default ⭐): Use "qwen2.5-coder:7b" (~4.5 GB VRAM — 65-75 tok/s, total VRAM 12.7 GB / 15.84 GB)
# - True  (Experimental): Use "qwen3-coder:8b"   (~5.5 GB VRAM — next-gen Qwen3 8B)
# ==============================================================================
USE_QWEN3_CODER = False

CONDUCTOR_MODEL = "deepseek-r1:8b"
WORKER_MODEL = "qwen3-coder:8b" if USE_QWEN3_CODER else "qwen2.5-coder:7b"

FAST_DUAL_MODELS = [
    (CONDUCTOR_MODEL, "🎼 Model 1 (Conductor): DeepSeek CoT <think> Fast Planning (~5.2 GB)"),
    (WORKER_MODEL,    "🛠️ Model 2 (Worker Specialist ⭐): High-Speed Coding, Workflow & Diagram (~4.5 GB)"),
]

print("⏳ Pulling Plan 2: Fast model pair (Dual-Resident T4)...")
for tag, role in FAST_DUAL_MODELS:
    print(f"\n📦 Pulling {tag} — {role}...")
    !ollama pull {tag}

print("\n🔥 Co-pinning both models into 16GB GPU VRAM on Tesla T4 (0.0s Switch)...")
for tag, _ in FAST_DUAL_MODELS:
    try:
        payload = json.dumps({"model": tag, "prompt": "hi", "stream": False, "keep_alive": "24h"}).encode("utf-8")
        req = urllib.request.Request("http://localhost:11434/api/generate", data=payload, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=120):
            print(f"  ✨ Successfully pinned to VRAM: {tag}")
    except Exception as e:
        print(f"  ⚠️ Preload note for {tag}: {e}")

clear_output(wait=True)
print(f"✅ DUAL-RESIDENT T4 SETUP COMPLETE! Both {CONDUCTOR_MODEL} and {WORKER_MODEL} are co-resident in 16GB VRAM (0.0s switch)!")
!ollama list

### Step 5: Expose Cloudflare Quick Tunnel & Connect to IDE
Establish an HTTPS tunnel forwarding `localhost:11434` to a public `*.trycloudflare.com` URL:
- **Tunnel URL:** Auto-extracted URL for configuring into your MCP client or IDE.
- **Pipe Drainer:** Background thread continuously flushes `cloudflared` stdout to prevent 64KB OS pipe buffer exhaustion and tunnel lockups (HTTP 524).

In [ ]:
import re
import subprocess
import threading
import time
from IPython.display import clear_output

# 1. Download and install Cloudflare Tunnel (cloudflared)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Launch cloudflared forwarding HTTPS to localhost:11434
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. Read stdout to extract public '*.trycloudflare.com' URL
tunnel_url = None
print("⏳ Establishing Cloudflare Quick Tunnel...")
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.2)

# 4. CRITICAL: Run background thread to drain stdout continuously and prevent 64KB OS pipe buffer lockups
def _drain_tunnel_logs(proc):
    try:
        for _ in iter(proc.stdout.readline, ""):
            pass
    except Exception:
        pass

threading.Thread(target=_drain_tunnel_logs, args=(tunnel_proc,), daemon=True).start()

clear_output(wait=True)

if tunnel_url:
    print("=" * 78)
    print("🎉 PLAN 2: FAST (T4 DUAL-RESIDENT) TUNNEL IS ONLINE!")
    print(f"🔗 PUBLIC ENDPOINT URL: {tunnel_url}")
    print("=" * 78)
    print("\n🚀 METHOD 1 (Paste endpoint URL directly into IDE / CLI chat):\n")
    print(f"   {tunnel_url}")
    print("   (The agent will automatically invoke `ollama_set_endpoint`, verify connectivity, and route calls to Colab GPU!)\n")
    print("⚡ METHOD 2 (Run one-line command in project terminal):\n")
    print(f"   python3 colab_ollama_mcp/server.py --set-endpoint {tunnel_url}")
    print("\n" + "-" * 78)
else:
    print("❌ Failed to obtain Cloudflare Tunnel URL. Please re-run this cell.")

### Step 6: Session Keep-Alive (HTML5 Silent Audio Loop)
Plays a continuous silent audio stream (500ms loop) to keep the browser tab in *Media Active* state, preventing runtime disconnection or tab throttling.

In [ ]:
from IPython.display import HTML, display

display(HTML('''
<div style="padding: 14px; background: #e8f5e9; border: 1px solid #4caf50; border-radius: 8px; font-family: sans-serif;">
  <b style="color: #2e7d32;">🟢 Keep-Alive Active (Silent Audio Loop)</b>
  <p style="margin: 4px 0 8px 0; color: #1b5e20; font-size: 13px;">
     Keeps your Google Colab session active and responsive while coding in your IDE.
  </p>
  <audio loop autoplay controls style="height: 30px;">
    <source src="data:audio/wav;base64,UklGRsQAAABXQVZFZm10IBAAAAABAAEAQB8AAIA+AAACABAAZGF0YaAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA" type="audio/wav">
  </audio>
</div>
'''))

### Step 7: System & VRAM Telemetry Dashboard (`/api/ps`)
Quick snapshot of GPU VRAM, RAM, Swap, Disk, and loaded models in VRAM:
- **`AUTO_REFRESH = False` (Default):** Instant single telemetry check.
- **`AUTO_REFRESH = True`:** Auto-refreshes every 5 seconds (`0 MB` DOM leak).

In [ ]:
import json
import shutil
import subprocess
import time
import urllib.request
from IPython.display import clear_output

AUTO_REFRESH = False
REFRESH_SECONDS = 5

def make_bar(pct: float, width: int = 24) -> str:
    filled = max(0, min(width, int(round((pct / 100.0) * width))))
    return "█" * filled + "░" * (width - filled)

def print_telemetry() -> None:
    print("=" * 78)
    print(f"📊 PLAN 2 FAST (T4 DUAL-RESIDENT) TELEMETRY — {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")
    print("=" * 78)

    # 1. GPU VRAM, Load & Temperature
    try:
        out = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=name,temperature.gpu,utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader,nounits",
        ], text=True).strip()
        for line in out.splitlines():
            name, temp, util, mem_used, mem_total = [x.strip() for x in line.split(",")]
            used_gb = float(mem_used) / 1024.0
            total_gb = float(mem_total) / 1024.0
            vram_pct = (float(mem_used) / float(mem_total)) * 100.0 if float(mem_total) > 0 else 0.0
            util_pct = float(util)
            print(f"🎮 GPU Hardware : {name}  (🌡️ {temp}°C)")
            print(f"⚡ GPU Core Load: [{make_bar(util_pct)}] {util_pct:5.1f}%")
            print(f"🧠 GPU VRAM     : [{make_bar(vram_pct)}] {vram_pct:5.1f}%  ({used_gb:.2f} GB / {total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ GPU Status   : {e}")

    # 2. System RAM, Page Cache & Swapfile
    try:
        meminfo = {}
        with open("/proc/meminfo", "r") as f:
            for line in f:
                parts = line.split(":")
                if len(parts) == 2:
                    meminfo[parts[0].strip()] = int(parts[1].strip().split()[0])
        total_ram_gb = meminfo.get("MemTotal", 0) / (1024 ** 2)
        avail_ram_gb = meminfo.get("MemAvailable", 0) / (1024 ** 2)
        used_ram_gb = total_ram_gb - avail_ram_gb
        ram_pct = (used_ram_gb / total_ram_gb * 100.0) if total_ram_gb > 0 else 0.0
        cached_gb = meminfo.get("Cached", 0) / (1024 ** 2)
        swap_total_gb = meminfo.get("SwapTotal", 0) / (1024 ** 2)
        swap_free_gb = meminfo.get("SwapFree", 0) / (1024 ** 2)
        swap_used_gb = swap_total_gb - swap_free_gb
        print(f"💻 System RAM   : [{make_bar(ram_pct)}] {ram_pct:5.1f}%  ({used_ram_gb:.2f} GB / {total_ram_gb:.2f} GB | Page Cache: {cached_gb:.2f} GB)")
        print(f"🛡️ NVMe Swapfile: {swap_used_gb:.2f} GB / {swap_total_gb:.2f} GB (Total Virtual Memory: {total_ram_gb + swap_total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ System RAM   : {e}")

    # 3. Colab SSD Disk Usage
    try:
        total_d, used_d, _ = shutil.disk_usage("/")
        used_d_gb = used_d / (1024 ** 3)
        total_d_gb = total_d / (1024 ** 3)
        disk_pct = (used_d / total_d * 100.0) if total_d > 0 else 0.0
        print(f"💾 Colab Disk   : [{make_bar(disk_pct)}] {disk_pct:5.1f}%  ({used_d_gb:.1f} GB / {total_d_gb:.1f} GB)")
    except Exception as e:
        print(f"⚠️ Colab Disk   : {e}")

    # 4. Active Models Pinned in GPU VRAM (/api/ps)
    print("-" * 78)
    try:
        with urllib.request.urlopen("http://localhost:11434/api/ps", timeout=5) as resp:
            ps_data = json.loads(resp.read().decode("utf-8"))
            running = ps_data.get("models", [])
            if running:
                print(f"🔥 Currently pinned in GPU VRAM ({len(running)} model{'s' if len(running) > 1 else ''}):")
                for m in running:
                    m_name = m.get("name", "unknown")
                    vram_gb = m.get("size_vram", 0) / (1024 ** 3)
                    total_gb = m.get("size", 0) / (1024 ** 3)
                    gpu_offload = (vram_gb / total_gb * 100.0) if total_gb > 0 else 100.0
                    print(f"   • {m_name} — VRAM: {vram_gb:.2f} GB ({gpu_offload:.0f}% GPU offload)")
            else:
                print("💤 GPU VRAM is currently idle (Models load on-demand on first request)")
    except Exception as e:
        print(f"🔴 Ollama Status: Offline or unresponsive ({e})")
    print("=" * 78)

try:
    while True:
        clear_output(wait=True)
        print_telemetry()
        if not AUTO_REFRESH:
            break
        print(f"🔄 Auto-refreshing every {REFRESH_SECONDS}s... (Click Stop ⏹️ to halt)")
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("⏹️ Telemetry dashboard stopped.")